# Notebook 10 — 1D-CNN auf Rohdaten (Deep Learning vs. Feature Engineering)

**Frage:** Kann ein Deep-Learning-Modell *seine eigenen* Features aus den rohen Sensordaten
lernen — und damit das handgemachte Feature Engineering aus NB09 schlagen?

**Aufbau:**
- Ein **1D-CNN** bekommt das **rohe 10-s-Fenster** (7 Kanäle: accel x/y/z, gyro x/y/z, magnitude),
  *keine* handgemachten Features. Es lernt die Repräsentation selbst.
- **Data Augmentation** (Jitter, Skalierung, Zeit-Verschiebung) gegen Overfitting bei wenig Daten.
- Fairer Vergleich: das CNN **und** die NB09-SVM (PLUS-Features) laufen auf **demselben**
  Session-Split (`GroupKFold(5)`) → Zahlen sind 1:1 vergleichbar.

**Hypothese (aus NB07/NB09):** Bei nur ~61 Sessions überfittet das CNN auf bekannte Sessions;
beim **cross-session** Split (GroupKFold) gewinnt vermutlich das Feature Engineering — weil die
handgemachten Features Domänenwissen als Prior mitbringen, das die fehlenden Daten ersetzt.

> Stufe 1 (Still vs. Essen) ist ~99 % — getestet wird die **Feinklassifikation** (Stufe 2).

---
## 1. Setup

In [ ]:
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch, find_peaks
from scipy.stats import kurtosis, skew

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cpu')   # kleines Modell, CPU reicht

CLASSES_RAW  = ['Apfel', 'Kaugummi', 'Skyr', 'Still', 'Essen']
CLASSES_FINE = ['Apfel', 'Kaugummi', 'Skyr', 'Essen']
TO_COARSE    = {'Apfel': 'Essen', 'Kaugummi': 'Essen', 'Skyr': 'Essen',
                'Still': 'Still', 'Essen': 'Essen'}

FS            = 50.0
TRIM_SECS     = 2
WINDOW_SECS   = 10.0    # wie Live-App / NB08
STEP_SECS     = 10.0    # kein Overlap -> sauberes Session-Splitting
MIN_TAIL_SECS = 8.0
K_ME          = 5
L             = 500     # feste CNN-Eingangslänge (10 s @ 50 Hz)
CHANNELS      = ['accelerationX', 'accelerationY', 'accelerationZ',
                 'rotationRateX', 'rotationRateY', 'rotationRateZ', 'magnitude']

print(f'PyTorch {torch.__version__}  |  Device: {DEVICE}  |  Kanäle: {len(CHANNELS)}  |  L={L}')

---
## 2. Daten pro Session laden

In [ ]:
DATA_DIR = Path('../data/raw')
_SKIP    = {'Metadata.csv', 'Annotation.csv'}

def preprocess(df):
    df = df.copy()
    t  = df['seconds_elapsed']
    df = df[(t >= t.iloc[0] + TRIM_SECS) & (t <= t.iloc[-1] - TRIM_SECS)].reset_index(drop=True)
    df['lin_x'] = df['accelerationX']
    df['lin_y'] = df['accelerationY']
    df['lin_z'] = df['accelerationZ']
    df['magnitude'] = np.sqrt(df['lin_x']**2 + df['lin_y']**2 + df['lin_z']**2)
    return df

sessions_per_class = {cls: [] for cls in CLASSES_RAW}
for zf in sorted(DATA_DIR.glob('*.zip')):
    for cls in CLASSES_RAW:
        if zf.name.startswith(cls + '_'):
            with zipfile.ZipFile(zf) as z:
                csv_name = next(f for f in z.namelist() if f.endswith('.csv') and f not in _SKIP)
                with z.open(csv_name) as f:
                    sessions_per_class[cls].append(preprocess(pd.read_csv(f)))
            break

print('Sitzungen pro Klasse:')
for cls in CLASSES_RAW:
    print(f'  {cls:10s}: {len(sessions_per_class[cls]):2d}')

---
## 3. Fenster → (a) Rohsignal fürs CNN  ·  (b) PLUS-Features für die SVM

Jedes 10-s-Fenster wird in **zwei** Repräsentationen abgelegt:
- **Roh:** alle 7 Kanäle auf feste Länge `L=500` resampled, pro Kanal z-standardisiert
  (entfernt Session-Offsets) → Eingang fürs CNN.
- **PLUS-Features:** die 52 handgemachten Features aus NB09 → Eingang für die Vergleichs-SVM.

So sehen beide Modelle **dieselben Fenster** — der Vergleich ist fair.

In [ ]:
def movement_mask(df):
    thr = max(0.02, K_ME * df['magnitude'].median())
    return df['magnitude'].rolling(50, center=True, min_periods=1).max() <= thr

def sliding_windows(df):
    t = df['seconds_elapsed'].values
    t_start, t_end = t[0], t[-1]
    out = []
    while t_start + MIN_TAIL_SECS <= t_end:
        w = df[(t >= t_start) & (t < t_start + WINDOW_SECS)].reset_index(drop=True)
        if len(w) > 1 and (w['seconds_elapsed'].iloc[-1] - w['seconds_elapsed'].iloc[0]) >= MIN_TAIL_SECS:
            out.append(w)
        t_start += STEP_SECS
    return out

# ── (a) Rohsignal: resample auf feste Länge + z-Standardisierung pro Kanal ─────
def window_to_raw(w):
    t  = w['seconds_elapsed'].values
    tn = np.linspace(t[0], t[-1], L)
    arr = np.stack([np.interp(tn, t, w[c].values) for c in CHANNELS], axis=0)  # (C, L)
    arr = (arr - arr.mean(axis=1, keepdims=True)) / (arr.std(axis=1, keepdims=True) + 1e-6)
    return arr.astype(np.float32)

# ── (b) Features: BASE (NB05) + NEU (NB09) ────────────────────────────────────
def extract_features_base(df):
    feats = {}
    for col in ['lin_x', 'lin_y', 'lin_z', 'magnitude']:
        feats[f'{col}_mean'] = df[col].mean(); feats[f'{col}_std'] = df[col].std(); feats[f'{col}_max'] = df[col].abs().max()
    feats['stillness_ratio'] = (df['magnitude'] < 0.02).mean()
    feats['movement_events'] = int((df['magnitude'] > df['magnitude'].quantile(0.75)).sum())
    for col in ['rotationRateX', 'rotationRateY', 'rotationRateZ']:
        feats[f'{col}_mean'] = df[col].mean(); feats[f'{col}_std'] = df[col].std(); feats[f'{col}_max'] = df[col].abs().max()
    for col in ['pitch', 'roll', 'yaw']:
        feats[f'{col}_mean'] = df[col].mean(); feats[f'{col}_std'] = df[col].std(); feats[f'{col}_range'] = df[col].max() - df[col].min()
    nperseg = min(256, len(df) // 2)
    freqs, psd = welch(df['magnitude'].values, fs=FS, nperseg=nperseg)
    chew = (freqs >= 0.5) & (freqs <= 4.0); cf, cp = freqs[chew], psd[chew]
    feats['total_power'] = float(psd.sum()); feats['chew_band_power'] = float(cp.sum())
    feats['rhythmicity'] = feats['chew_band_power'] / feats['total_power'] if feats['total_power'] > 0 else 0.0
    feats['dominant_chew_freq'] = float(cf[np.argmax(cp)]) if len(cp) > 0 else 0.0
    return feats

NEW_FEATURES = ['mag_kurtosis', 'mag_skew', 'crest_factor', 'jerk_mean', 'jerk_std',
                'spectral_entropy', 'spectral_centroid', 'spectral_bandwidth',
                'band_power_slow', 'band_power_mid', 'band_power_fast', 'high_freq_power',
                'ac_peak_height', 'chew_freq_ac', 'chews_per_sec', 'inter_chew_cv']

def extract_new_features(df):
    f = {k: 0.0 for k in NEW_FEATURES}
    mag = df['magnitude'].values; t = df['seconds_elapsed'].values
    dur = (t[-1] - t[0]) if len(t) > 1 else 0.0
    rms = np.sqrt(np.mean(mag**2)) if len(mag) else 0.0
    if len(mag) > 3: f['mag_kurtosis'] = float(kurtosis(mag))
    if len(mag) > 2: f['mag_skew'] = float(skew(mag))
    if rms > 1e-9:   f['crest_factor'] = float(np.max(np.abs(mag)) / rms)
    if len(mag) > 2:
        jerk = np.diff(mag) * FS; f['jerk_mean'] = float(np.mean(np.abs(jerk))); f['jerk_std'] = float(np.std(jerk))
    nperseg = min(256, len(mag) // 2)
    if nperseg >= 2:
        freqs, psd = welch(mag, fs=FS, nperseg=nperseg); tot = psd.sum()
        if tot > 0:
            p = psd / tot; pp = p[p > 0]
            f['spectral_entropy'] = float(-np.sum(pp * np.log2(pp)) / np.log2(len(p)))
            f['spectral_centroid'] = float(np.sum(freqs * p))
            f['spectral_bandwidth'] = float(np.sqrt(np.sum((freqs - f['spectral_centroid'])**2 * p)))
            band = lambda lo, hi: float(psd[(freqs >= lo) & (freqs < hi)].sum() / tot)
            f['band_power_slow'] = band(0.5, 1.5); f['band_power_mid'] = band(1.5, 2.5)
            f['band_power_fast'] = band(2.5, 4.0); f['high_freq_power'] = band(4.0, 15.0)
    sig = mag - mag.mean() if len(mag) else mag
    if len(sig) > int(FS):
        ac = np.correlate(sig, sig, mode='full')[len(sig) - 1:]; ac = ac / (ac[0] + 1e-12)
        lo, hi = int(FS / 4.0), min(int(FS / 0.5), len(ac) - 1)
        if hi > lo:
            peak = np.argmax(ac[lo:hi]) + lo; f['ac_peak_height'] = float(ac[peak]); f['chew_freq_ac'] = float(FS / peak)
    if len(mag) > 5 and mag.std() > 0:
        peaks, _ = find_peaks(mag, distance=max(1, int(FS * 0.2)), prominence=mag.std())
        if dur > 0: f['chews_per_sec'] = float(len(peaks) / dur)
        if len(peaks) > 1:
            iv = np.diff(t[peaks])
            if iv.mean() > 0: f['inter_chew_cv'] = float(iv.std() / iv.mean())
    return f

print('Fenster-Funktionen bereit (window_to_raw + PLUS-Features)')

---
## 4. Datensatz bauen (nur Essen-Subset = Stufe 2)

In [ ]:
raw_list, feat_list, y_list, grp_list = [], [], [], []
session_counter = 0
for cls in CLASSES_RAW:
    for df in sessions_per_class[cls]:
        if TO_COARSE[cls] != 'Essen':     # nur Essen-Klassen (Stufe 2)
            session_counter += 1
            continue
        for w in sliding_windows(df):
            clean = w[movement_mask(w)].reset_index(drop=True)
            if len(clean) > 30:
                raw_list.append(window_to_raw(w))
                feat_list.append({**extract_features_base(clean), **extract_new_features(clean)})
                y_list.append(cls)
                grp_list.append(session_counter)
        session_counter += 1

X_raw  = np.stack(raw_list)                       # (N, C, L)
X_feat = pd.DataFrame(feat_list).values           # (N, 52)
y_eat  = np.array(y_list)
grp    = np.array(grp_list)

print(f'Essen-Fenster: {len(X_raw)}  über {len(np.unique(grp))} Sessions')
print(f'  Roh-Tensor: {X_raw.shape}   Feature-Matrix: {X_feat.shape}')
for c in CLASSES_FINE:
    print(f'  {c:10s}: {(y_eat==c).sum():3d}')

---
## 5. 1D-CNN + Data Augmentation

Klein gehalten (BatchNorm + Dropout) gegen Overfitting. Augmentation wird **on-the-fly**
nur auf Trainings-Batches angewandt.

In [ ]:
class ChewCNN(nn.Module):
    def __init__(self, n_ch, n_cls):
        super().__init__()
        self.feat = nn.Sequential(
            nn.Conv1d(n_ch, 16, 7, padding=3), nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(16, 32, 5, padding=2),   nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(32, 64, 3, padding=1),   nn.BatchNorm1d(64), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(64, 32), nn.ReLU(),
                                  nn.Dropout(0.3), nn.Linear(32, n_cls))
    def forward(self, x):
        return self.head(self.feat(x))

def augment(x):
    """x: (B, C, L). Jitter + Amplituden-Skalierung + zirkuläre Zeit-Verschiebung."""
    x = x + 0.05 * torch.randn_like(x)
    x = x * (1.0 + 0.10 * torch.randn(x.size(0), 1, 1))
    x = torch.roll(x, shifts=int(torch.randint(-25, 26, (1,)).item()), dims=2)
    return x

def train_cnn(Xtr, ytr, n_cls, augment_on, epochs=40):
    torch.manual_seed(SEED)
    model = ChewCNN(Xtr.shape[1], n_cls).to(DEVICE)
    counts = np.bincount(ytr, minlength=n_cls).astype(float)
    cw = torch.tensor(len(ytr) / (n_cls * np.maximum(counts, 1)), dtype=torch.float32)
    crit = nn.CrossEntropyLoss(weight=cw)
    opt  = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    dl = DataLoader(TensorDataset(torch.tensor(Xtr), torch.tensor(ytr, dtype=torch.long)),
                    batch_size=32, shuffle=True)
    model.train()
    for _ in range(epochs):
        for xb, yb in dl:
            if augment_on:
                xb = augment(xb)
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
    return model

def predict_cnn(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X)).argmax(1).cpu().numpy()

print('ChewCNN + Trainings-/Augmentation-Funktionen bereit')

---
## 6. Cross-Session-Vergleich (GroupKFold = 5, je Session zusammengehalten)

Drei Modelle auf **identischen** Splits:
1. **CNN (roh)** ohne Augmentation
2. **CNN (roh) + Augmentation**
3. **SVM (PLUS-Features)** — der NB09-Champion

In [ ]:
gkf = GroupKFold(n_splits=5)
le  = LabelEncoder().fit(CLASSES_FINE)
yint = le.transform(y_eat)

oof = {'CNN': np.empty(len(y_eat), dtype=object),
       'CNN+Aug': np.empty(len(y_eat), dtype=object),
       'SVM-PLUS': np.empty(len(y_eat), dtype=object)}

for fold, (tr, te) in enumerate(gkf.split(X_raw, yint, groups=grp), 1):
    print(f'Fold {fold}/5  (train={len(tr)}, test={len(te)})…', flush=True)

    m  = train_cnn(X_raw[tr], yint[tr], len(CLASSES_FINE), augment_on=False)
    oof['CNN'][te] = le.inverse_transform(predict_cnn(m, X_raw[te]))

    m2 = train_cnn(X_raw[tr], yint[tr], len(CLASSES_FINE), augment_on=True)
    oof['CNN+Aug'][te] = le.inverse_transform(predict_cnn(m2, X_raw[te]))

    pipe = Pipeline([('sc', StandardScaler()),
                     ('svm', SVC(kernel='rbf', C=10, class_weight='balanced', random_state=SEED))])
    pipe.fit(X_feat[tr], y_eat[tr])
    oof['SVM-PLUS'][te] = pipe.predict(X_feat[te])

print('\nCross-Session-Accuracy (GroupKFold, Stufe 2):')
acc = {k: accuracy_score(y_eat, v) for k, v in oof.items()}
for k in ['CNN', 'CNN+Aug', 'SVM-PLUS']:
    print(f'  {k:10s}: {acc[k]:.1%}')

---
## 7. Ergebnisse & Interpretation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Links: Balkenvergleich
ax = axes[0]
order = ['CNN', 'CNN+Aug', 'SVM-PLUS']
colors = ['#bab0ac', '#e15759', '#4e79a7']
vals = [acc[k] for k in order]
bars = ax.bar(order, vals, color=colors, edgecolor='white')
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.008, f'{v:.1%}', ha='center', va='bottom', fontweight='bold')
ax.axhline(0.25, ls='--', color='gray', lw=1, label='Zufall (4 Klassen)')
ax.set_ylim(0, 1.0); ax.set_ylabel('Cross-Session-Accuracy (GroupKFold)')
ax.set_title('Deep Learning vs. Feature Engineering', fontweight='bold')
ax.legend()

# Rechts: Confusion Matrix bestes CNN
best_cnn = 'CNN+Aug' if acc['CNN+Aug'] >= acc['CNN'] else 'CNN'
cm = confusion_matrix(y_eat, oof[best_cnn], labels=CLASSES_FINE)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=CLASSES_FINE,
            yticklabels=CLASSES_FINE, ax=axes[1], linewidths=0.5, linecolor='white',
            annot_kws={'size': 11, 'weight': 'bold'}, vmin=0)
axes[1].set_title(f'{best_cnn}  ({acc[best_cnn]:.1%})', fontweight='bold')
axes[1].set_xlabel('Vorhergesagt'); axes[1].set_ylabel('Tatsächlich')

plt.suptitle('NB10 — 1D-CNN auf Rohdaten  |  Stufe 2  |  10-s-Fenster', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/images/nb10_cnn_vs_features.png', dpi=150, bbox_inches='tight')
plt.show()

print('Detailreport bestes CNN:')
print(classification_report(y_eat, oof[best_cnn], labels=CLASSES_FINE, zero_division=0))

---
## 8. Fazit (nach dem Lauf ausfüllen)

- **CNN vs. SVM-PLUS:** Wer gewinnt beim *cross-session* Split?
- **Bringt Augmentation etwas?** (CNN+Aug − CNN) — schließt sie den Abstand zur SVM?
- **Erwartung (NB07/NB09):** Bei ~61 Sessions reicht die Datenvielfalt nicht, damit das CNN
  bessere Features lernt als die handgebauten. Augmentation hilft etwas, ersetzt aber keine
  echten neuen Sessions.

**Take-away für die Präsentation:**
> "Wir haben Deep Learning auf Rohdaten getestet. Ein CNN *kann* prinzipiell eigene Features
> lernen — aber bei unserer Datenmenge schlägt Feature Engineering mit Domänenwissen das CNN.
> Der Engpass ist die **Session-/Personen-Vielfalt**, nicht die Modellkomplexität."

→ Konsistent mit NB07 (LOSO-Lücke) und NB09 (Domänen-Features verbessern Generalisierung).